# ResNet18 Benchmark

Only support PY39, OR PY38

In [1]:
model_name = "ResNet18"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"

import tensorflow as tf
import numpy as np
import iree.compiler
import iree.runtime

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
    
def get_dataframe(forward, backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": forward,
            "pass": "Forward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
        pd.DataFrame({
            "time": forward + backward,
            "pass": "Full",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=100):
    return ti(stmt, globals=globals(), number=n) * 1000 / n

2024-03-20 12:52:57.462566: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1


## Experimental

In [ ]:
# with tf.device('/device:XLA_GPU:0'):
# with tf.device('/CPU:0'):
with tf.device('/device:XLA_GPU:0'):
    image = tf.random.normal((1, 3, 224, 224))
    model = tf.function(tf.keras.applications.resnet50.ResNet50(
        include_top=False,
        weights='imagenet',
        input_tensor=image,
        input_shape=(3, 224, 224),
        pooling='avg',
        classes=1000
    ))

    with tf.GradientTape(persistent=True) as tape:
        tape.watch(image)
        output = model(image)

grad = tf.random.normal((1, 1000))

#baseline_f = []
#for i in range(11):
#  baseline_f += [timeit("model(image)") / 17.0]

#baseline_b = []
#for i in range(11):
#  baseline_f += [timeit("tape.gradient(output, image, grad)") / 17.0]

#print(baseline_f)
#print(baseline_b)
#baseline_f = np.mean(baseline_f)
#baseline_b = np.mean(baseline_b)

baseline_f = timeit("model(image)")
baseline_b = timeit("tape.gradient(output, image, grad)")

### TensorFlow (Baseline)

In [ ]:
df = pd.DataFrame()
df = pd.concat([df, get_dataframe(baseline_f, baseline_b, "TensorFlow")])
print(df)

## Result

In [ ]:
df.to_csv(f"{model_name}-time.csv")
df.style.hide(axis="index")

In [ ]:
plt.rcParams["figure.dpi"] = 300

sns.barplot(df, x="pass", y="time", hue="item")
plt.xlabel("Pass")
plt.ylabel("Time (ms)")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-time.png")

In [ ]:
forward = df[df["pass"] == "Forward"]
forward["acceleration"] = baseline_f / forward["time"]
forward

In [ ]:
backward = df[df["pass"] == "Backward"]
backward["acceleration"] = baseline_b / backward["time"]
backward

In [ ]:
full = df[df["pass"] == "Full"]
full["acceleration"] = (baseline_b + baseline_f) / full["time"]
full

In [ ]:
df = pd.concat([forward, backward, full])
df.to_csv(f"{model_name}-acceleration.csv")

sns.barplot(df, x="pass", y="acceleration", hue="item")
plt.xlabel("Pass")
plt.ylabel("Acceleration")
plt.legend().set_title("Item")
plt.title(model_name)

plt.savefig(f"{model_name}-acceleration.png")